### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
sys.path.append('./utils')

### Random seed for reproducibility

In [2]:
import torch
import random
import numpy as np
#import multiprocessing as mp
#mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc
import svg_constraints 
from svg_processor import SVGSanitizer, SVGProcessor

class Model:
    
    def __init__(self):

        self.model_path="./lora/Qwen3_4B_lora_fp16_r256_s60000_e2_msl2048"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            #quantization="AWQ",
            gpu_memory_utilization=0.95,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )
       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded_list = self.get_response(descriptions)
        final_svg_code_list = []
    
        for description, output in zip(descriptions, output_decoded_list):
            base_svg = SVGProcessor.clean_and_extract_svgs(output, self.default_svg)
            clean_svg = self.sanitizer.enforce_constraints(base_svg)
            final_svg = SVGProcessor.svg_conversion_check(description, clean_svg, self.default_svg)
            final_svg_code_list.append(final_svg)
    
        return final_svg_code_list


INFO 05-16 17:25:10 [__init__.py:239] Automatically detected platform cuda.


In [4]:
model=Model()

WARNING 05-16 17:25:11 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 05-16 17:25:16 [config.py:585] This model supports multiple tasks: {'generate', 'classify', 'embed', 'reward', 'score'}. Defaulting to 'generate'.
INFO 05-16 17:25:16 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-16 17:25:17 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/Qwen3_4B_lora_fp16_r256_s60000_e2_msl2048', speculative_config=None, tokenizer='./lora/Qwen3_4B_lora_fp16_r256_s60000_e2_msl2048', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 05-16 17:25:21 [loader.py:447] Loading weights took 3.22 seconds
INFO 05-16 17:25:21 [gpu_model_runner.py:1186] Model loading took 7.5454 GB and 3.523851 seconds
INFO 05-16 17:25:23 [kv_cache_utils.py:566] GPU KV cache size: 8,192 tokens
INFO 05-16 17:25:23 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 8.00x
INFO 05-16 17:25:28 [gpu_model_runner.py:1534] Graph capturing finished in 5 secs, took 0.15 GiB
INFO 05-16 17:25:28 [core.py:151] init engine (profile, create kv cache, warmup model) took 7.07 seconds


In [5]:
#model.predict(['who are you?'])

In [6]:
import sys
sys.path.append(r'/home/vino/ML_Projects/Drawing_with_LLMs/drawing-with-llms')
import pandas as pd

df1=pd.read_csv(r'./drawing-with-llms/test_filtered_1_batch_vqa_gpt4.csv',header=[0])
df2=pd.read_csv(r'./drawing-with-llms/description_master_test_gemini_25pro_2k.csv',header=[0])
#df3=pd.read_csv(r'./drawing-with-llms/gemini_25_pro_validation/train_filtered_1_batch_gpt4.csv',header=[0])
#print(df3.shape)
#df2=df2.drop_duplicates(['description'])
#df=pd.concat([df1[['description']],df2['description']],axis=0)
df=df2.drop_duplicates(['description'])
df=df.iloc[:300]
print(df.shape)
df.head(2)

(300, 1)


,description
0,Sunrise over a misty valley
1,Overlapping translucent circles in pastel shades


In [7]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [8]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 15
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


Batch prediction:   0%|                                  | 0/20 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:04<01:06,  4.72s/it, est. speed input: 13.34
cessed prompts:  13%|▏| 2/15 [00:07<00:43,  3.31s/it, est. speed input: 18.15
cessed prompts:  27%|▎| 4/15 [00:08<00:18,  1.64s/it, est. speed input: 30.58
cessed prompts:  33%|▎| 5/15 [00:11<00:21,  2.14s/it, est. speed input: 27.50
cessed prompts:  40%|▍| 6/15 [00:11<00:14,  1.56s/it, est. speed input: 32.30
cessed prompts:  47%|▍| 7/15 [00:13<00:13,  1.69s/it, est. speed input: 32.52
cessed prompts:  53%|▌| 8/15 [00:14<00:08,  1.26s/it, est. speed input: 36.47
cessed prompts:  60%|▌| 9/15 [00:17<00:11,  1.94s/it, est. speed input: 32.80
cessed prompts:  67%|▋| 10/15 [00:20<00:10,  2.16s/it, est. speed input: 31.5
cessed prompts:  80%|▊| 12/15 [00:27<00:08,  2.76s/it, est. speed input: 28.1
cessed prompts:  87%|▊| 13/15 [00:34<00:07,  3.80s/it, est. s

In [9]:
df['svg_3']=results

In [10]:
model.close_model()

In [11]:
from siglip_class import SVGMetricEvaluator
from aesthetic_evaluator import AestheticEvaluator

In [12]:
#SigLip Score
from tqdm import tqdm
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
100%|█████████████████████████████████████████| 300/300 [00:18<00:00, 16.47it/s]


In [13]:
#Aes Score
from tqdm import tqdm
tqdm.pandas()
aes_eval = AestheticEvaluator()
df['aes_score_3'] = df.progress_apply(lambda row: aes_eval.get_score(row['svg_3']), axis=1)

100%|█████████████████████████████████████████| 300/300 [00:29<00:00, 10.26it/s]


In [14]:
#combined score
df['combined_score_3'] = (df['svg_score_3']+df['svg_score_3']+df['aes_score_3'])/3

In [15]:
print('mean_svg_score:',df['svg_score_3'].mean(),'mean_aes_score:',df['aes_score_3'].mean(),'combined_score:',df['combined_score_3'].mean())

mean_svg_score: 0.13183723937156713 mean_aes_score: 0.4366082084973653 combined_score: 0.23342756241349985


In [16]:
default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
df_default_svg=df[df['svg_3']==default_svg]
print('default_svg_count:',df_default_svg.shape[0])
print('default_svg_score_mean:',df_default_svg['svg_score_3'].mean(),'default_aes_score_mean:',df_default_svg['aes_score_3'].mean(),\
     'combined_score:',df_default_svg['combined_score_3'].mean())

default_svg_count: 20
default_svg_score_mean: 5.833727232226922e-07 default_aes_score_mean: 0.4369946956634522 combined_score: 0.1456652874696329


In [17]:
df_non_default_svg=df[df['svg_3']!=default_svg]
print('non-default_svg_count:',df_non_default_svg.shape[0])
print('non-default_svg_score_mean:',df_non_default_svg['svg_score_3'].mean(),\
      'non-default_aes_score_mean:',df_non_default_svg['aes_score_3'].mean(),\
        'combined_score:',df_non_default_svg['combined_score_3'].mean())

non-default_svg_count: 280
non-default_svg_score_mean: 0.14125414337148456 non-default_aes_score_mean: 0.43658060227121626 combined_score: 0.2396962963380618


In [18]:
df['svg_3'].iloc[2]

'<svg xmlns="http://www.w3.org/2000/svg" width="200" height="200" viewBox="0 0 200 200"><defs><linearGradient id="sandGradient" x1="0%" y1="0%" x2="100%" y2="100%"><stop offset="0%"/><stop offset="100%"/></linearGradient></defs><path d="M 50 20 L 150 20 L 170 50 L 170 150 L 30 150 L 30 50 L 50 20 Z" fill="url(#sandGradient)" stroke="none" stroke-width="2"/><rect x="60" y="30" width="80" height="20" fill="none" stroke="rgb(230,200,150)" stroke-width="1"/><rect x="60" y="140" width="80" height="20" fill="none" stroke="rgb(230,200,150)" stroke-width="1"/><line x1="100" y1="50" x2="100" y2="150" stroke="rgb(230,200,150)" stroke-width="1" stroke-dasharray="5,5"/></svg>'